# Submission 1 for BirdCLEF+ 2026 Sound Classification Kaggle competition.

In [13]:
import pandas as pd
import numpy as np
import librosa
from pathlib import Path
from PIL import Image
from fastai.vision.all import load_learner
import torch

In [14]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('birdclef-2026')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/birdclef-2026


In [15]:
!ls $path'/test_soundscapes'

readme.txt


In [17]:
# ---- Config ----
MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/only_top10_most_common_species/1/export.pkl'
TEST_DIR = Path(path + '/test_soundscapes')
TARGET_SIZE = (224, 224)
CLIP_DURATION = 5
# test soundscapes are resampled to 32khz per the dataset description
SAMPLE_RATE = 32000

# ---- All 234 species columns (from sample_submission.csv) ----
sample_sub = pd.read_csv(path + '/sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']

# ---- Load model ----
learn = load_learner(MODEL_PATH)
top_10_classes = learn.dls.vocab  # the 10 species the model knows

# ---- Helper: audio -> spectrogram image tensor ----
def audio_to_img(samples):
    S = librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE)
    S_db = librosa.power_to_db(S, ref=np.max)
    S_norm = ((S_db - S_db.min()) / (S_db.max() - S_db.min()) * 255).astype(np.uint8)
    img = Image.fromarray(S_norm).resize(TARGET_SIZE)
    return img

# ---- Run inference ----
rows = []

test_files = list(TEST_DIR.glob('*.ogg'))
print(f"Found {len(test_files)} test soundscape files")
print(test_files[:3])  # show a few to check the paths look right

for soundscape in TEST_DIR.glob('*.ogg'):
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    clip_length = CLIP_DURATION * SAMPLE_RATE
    
    start = 0
    end_time = CLIP_DURATION  # tracks the end second of each segment
    
    while start + clip_length <= len(samples):
        chunk = samples[start:start + clip_length]
        img = audio_to_img(chunk)
        
        # Run through model
        pred_class, pred_idx, probs = learn.predict(img)
        probs = probs.numpy()
        
        # Build a dict with 0.0 for all 234 species
        row = {species: 0.0 for species in all_species}
        
        # Fill in probabilities for the 10 species the model knows
        for i, species in enumerate(top_10_classes):
            if species in row:
                row[species] = float(probs[i])
        
        # row_id format: {soundscape_stem}_{end_time}
        row['row_id'] = f"{soundscape.stem}_{end_time}"
        rows.append(row)
        
        start += clip_length
        end_time += CLIP_DURATION

# ---- Save submission ----
submission = pd.DataFrame(rows)
submission = submission[['row_id'] + all_species]  # enforce column order
submission.to_csv('submission.csv', index=False)
print(f"Done. {len(submission)} rows written.")


Found 0 test soundscape files
[]


/usr/local/lib/python3.12/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


KeyError: "None of [Index(['row_id', '1161364', '116570', '1176823', '1491113', '1595929',\n       '209233', '22930', '22956', '22961',\n       ...\n       'whnjay1', 'whtdov', 'whwpic1', 'y00678', 'yebcar', 'yebela1', 'yecmac',\n       'yecpar', 'yehcar1', 'yeofly1'],\n      dtype='object', length=235)] are in the [columns]"